# Day 54 Practice: Parsing, Validation and Mocks

Work through the exercises in order. Nothing here needs an API key. `review_analyser.py` and `mock_model.py` in
this folder contain the demo's finished code; some exercises import from them.

✅ Exercises 1–4: everyone · Exercise 5: most of you · Exercise 6: challenge

In [ ]:
import json

from mock_model import MockLLM
from review_analyser import (ALLOWED_SENTIMENTS, analyse_review, build_prompt, build_strict_prompt,
                             parse_json_reply, validate_analysis)

## Exercise 1: Will it parse?

For each text, predict: does `json.loads` succeed? If yes, what **type** is the result? If no, why not?

| | Text | Your prediction |
|-|------|-----------------|
| a | `{"rating": 5, "verified": true}` | |
| b | `{'rating': 5}` | |
| c | `[1, 2, 3]` | |
| d | `"positive"` (with the double quotes) | |
| e | `Sure! {"rating": 5}` | |
| f | `{"rating": 5,}` | |

Then call `try_parse` on each text to check.

In [ ]:
def try_parse(text):
    """Print what json.loads makes of text, or the error."""
    try:
        value = json.loads(text)
        print("OK ->", type(value), value)
    except json.JSONDecodeError as error:
        print("JSONDecodeError ->", error)


texts = ['{"rating": 5, "verified": true}', "{'rating': 5}", '[1, 2, 3]', '"positive"',
         'Sure! {"rating": 5}', '{"rating": 5,}']

# your code here

## Exercise 2: Find the JSON

Write `find_json(reply)`: return the JSON object inside `reply` (from the first `{` to the **last** `}`), parsed
with `json.loads`. If there is no `{` or `}`, raise `ValueError("no JSON object found")`.

Expected output of the checks:
```
{'a': 1}
{'a': 1}
{'a': {'b': 2}}
ValueError: no JSON object found
```
Why does the third check need the **last** `}` and not the first?

In [ ]:
# your code here


replies = ['{"a": 1}', 'Sure!\n```json\n{"a": 1}\n```', 'Here: {"a": {"b": 2}} Thanks!', "No JSON here"]
# for reply in replies:
#     try:
#         print(find_json(reply))
#     except ValueError as error:
#         print("ValueError:", error)

## Exercise 3: Validate a support ticket

A shop asks a model to sort support emails. A valid reply has exactly these fields:

| Key | Rule |
|-----|------|
| `category` | text, one of `billing`, `technical`, `delivery`, `other` (any upper/lower case, spaces around allowed) |
| `urgent` | `true` or `false` (a bool, not the text `"yes"`) |
| `summary` | text, not empty |

Write `validate_ticket(data)` that raises `ValueError` with a helpful message at the first problem, or returns a
clean copy (`category` stripped and lowercase, `summary` stripped).

Expected output (your messages may be worded differently):
```
OK: {'category': 'billing', 'urgent': True, 'summary': 'Charged twice.'}
Rejected: urgent must be true or false
Rejected: invalid category: complaint
Rejected: missing key: summary
```

In [ ]:
CATEGORIES = ["billing", "technical", "delivery", "other"]

# your code here


tickets = [
    '{"category": " Billing", "urgent": true, "summary": "Charged twice. "}',
    '{"category": "delivery", "urgent": "yes", "summary": "Parcel lost."}',
    '{"category": "complaint", "urgent": false, "summary": "Rude driver."}',
    '{"category": "technical", "urgent": false}',
]
# for text in tickets:
#     try:
#         print("OK:", validate_ticket(json.loads(text)))
#     except ValueError as error:
#         print("Rejected:", error)

## Exercise 4: Fix the analyser

This function has **four** problems (one of them only shows up when the model keeps misbehaving). Find them, then
write a fixed `analyse(review, ask)` in the cell below. Use `build_prompt`, `build_strict_prompt`,
`parse_json_reply` and `validate_analysis` from `review_analyser.py`. It must try **at most twice** and return
`None` if both replies are unusable.

```python
def analyse(review, ask):
    prompt = build_prompt(review)
    while True:
        reply = ask(prompt)
        try:
            data = json.load(reply)
            if data["sentiment"] not in ALLOWED_SENTIMENTS:
                raise ValueError("bad sentiment")
            return data
        except KeyError:
            prompt = build_strict_prompt(review)
```

Problems I found:
1. ...
2. ...
3. ...
4. ...

Expected output of the checks:
```
{'sentiment': 'negative', 'topics': ['quality'], 'summary': 'Strap ripped on the first day.'} 1
None 2
```

In [ ]:
# fixed version


# good_model = MockLLM()
# print(analyse("Strap ripped on the first day. Really poor stitching.", good_model.ask), good_model.calls)
# broken_model = MockLLM(fail_first=10)
# print(analyse("Strap ripped on the first day. Really poor stitching.", broken_model.ask), broken_model.calls)

## Exercise 5: Test with `MockLLM`

Write **four** test functions for `analyse_review` (imported from `review_analyser.py`), each with its own new
`MockLLM`:

1. `test_first_try`: a good model gives a result with the expected sentiment after **1** call.
2. `test_one_retry`: `MockLLM(fail_first=1)` still gives a result, after **2** calls.
3. `test_gives_up`: `MockLLM(fail_first=3)` gives `None` after **2** calls (not 3!).
4. `test_review_is_inside_tags`: the first prompt the model received contains `<review>` and the review text.

Call all four and print `all tests passed`. (Homework: move them into `test_my_analyser.py` and run
`uv run pytest -q` in the terminal.)

In [ ]:
# your code here

## Exercise 6 (challenge): A scripted mock

`MockLLM` can only produce one kind of bad reply. For exact test scenarios, write a class `ScriptedLLM` that
receives a **list of replies** and returns them one by one, in order (and counts its calls). Then use it to show:

a) a reply wrapped in a sentence, with values in mixed case (`"Positive"`, `["Battery"]`), is cleaned;
b) first reply with `"sentiment": "neutral"`, second reply valid → the **second** reply's values are returned;
c) two replies that are not JSON → `None`.

Expected output (the `Unusable reply` lines are printed by `analyse_review`):
```
a) {'sentiment': 'positive', 'topics': ['battery'], 'summary': 'Lasts all day.'}
Unusable reply: invalid sentiment: neutral
b) {'sentiment': 'negative', 'topics': ['delivery'], 'summary': 'Arrived late.'} calls: 2
Unusable reply: no JSON object found in the reply
Unusable reply: no JSON object found in the reply
c) None
```

In [ ]:
# your code here